# Assignment 1

AME 5052 - Applied Machine Learning Lab

Run the cells from top to bottom. The examples use built-in scikit-learn datasets. Output tables print below each cell and plots display inline.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score, confusion_matrix, classification_report

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Linear regression with Ridge and Lasso regularization

In [ ]:
print("\n1(a) Linear Regression, Ridge, and Lasso")
diabetes = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.25, random_state=42
)
regressors = {
    "Linear Regression": LinearRegression(),
    "Ridge (alpha=1.0)": Ridge(alpha=1.0),
    "Lasso (alpha=0.1)": Lasso(alpha=0.1, max_iter=10000),
}
regression_rows = []
prediction_plot = None
for name, estimator in regressors.items():
    model = make_pipeline(StandardScaler(), estimator)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    regression_rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions),
    })
    if name == "Linear Regression":
        prediction_plot = predictions
regression_results = pd.DataFrame(regression_rows).round(3)
print(regression_results.to_string(index=False))
plt.figure(figsize=(5.5, 4))
plt.scatter(y_test, prediction_plot, alpha=0.75)
low, high = min(y_test.min(), prediction_plot.min()), max(y_test.max(), prediction_plot.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual target")
plt.ylabel("Predicted target")
plt.title("Linear Regression: Actual vs Predicted")
plt.tight_layout()
plt.show()

## Decision tree: Gini, entropy, and information gain

In [ ]:
print("\n1(b) Decision Tree splitting criteria")
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.25, random_state=42, stratify=iris.target
)
tree_scores = {}
for criterion in ["gini", "entropy"]:
    model = DecisionTreeClassifier(criterion=criterion, max_depth=4, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    tree_scores[criterion] = accuracy_score(y_test, pred)
    print(f"{criterion.title()} tree accuracy: {tree_scores[criterion]:.3f}")

def entropy(labels):
    """Calculate Shannon entropy for a set of class labels."""
    _, counts = np.unique(labels, return_counts=True)
    probabilities = counts / counts.sum()
    return float(-np.sum(probabilities * np.log2(probabilities)))

feature = iris.data[:, 2]
threshold = float(np.median(feature))
left_labels = iris.target[feature <= threshold]
right_labels = iris.target[feature > threshold]
weighted_child_entropy = (
    len(left_labels) / len(iris.target) * entropy(left_labels)
    + len(right_labels) / len(iris.target) * entropy(right_labels)
)
info_gain = entropy(iris.target) - weighted_child_entropy
print(f"Example split threshold on petal length: {threshold:.2f}")
print(f"Information gain for that split: {info_gain:.3f}")
plt.figure(figsize=(5.5, 3.8))
plt.bar(["Gini", "Entropy"], [tree_scores["gini"], tree_scores["entropy"]])
plt.ylim(0, 1.05)
plt.ylabel("Test accuracy")
plt.title("Decision Tree Criteria")
plt.show()

## Logistic regression: sigmoid, decision boundary, Ridge/L2 and Lasso/L1

In [ ]:
print("\n1(c) Logistic Regression with L1 and L2 regularization")
cancer = load_breast_cancer()
X = cancer.data
# Set label 1 to malignant so recall has a clear medical interpretation.
y = (cancer.target == 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
def logistic_estimator(penalty):
    # Compatible with older scikit-learn and the newer l1_ratio API.
    if LogisticRegression().get_params().get("penalty") == "deprecated":
        if penalty == "l1":
            return LogisticRegression(l1_ratio=1.0, C=1.0, solver="saga", max_iter=5000, random_state=42)
        return LogisticRegression(l1_ratio=0.0, C=1.0, solver="lbfgs", max_iter=3000, random_state=42)
    return LogisticRegression(penalty=penalty, C=1.0, solver="liblinear", max_iter=3000, random_state=42)

logistic_models = {
    "L2 / Ridge-style": logistic_estimator("l2"),
    "L1 / Lasso-style": logistic_estimator("l1"),
}
logistic_rows = []
for name, estimator in logistic_models.items():
    model = make_pipeline(StandardScaler(), estimator)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    logistic_rows.append({"Model": name, "Accuracy": accuracy_score(y_test, pred), "F1": f1_score(y_test, pred)})
    print(f"{name}: accuracy={accuracy_score(y_test, pred):.3f}, malignant-class F1={f1_score(y_test, pred):.3f}")

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 300)
plt.figure(figsize=(5.5, 3.8))
plt.plot(z, sigmoid(z))
plt.axhline(0.5, linestyle="--", linewidth=1)
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Linear score (z)")
plt.ylabel("Predicted probability")
plt.title("Sigmoid Function and 0.5 Decision Threshold")
plt.tight_layout()
plt.show()
print("At threshold 0.5, the decision changes at sigmoid input z = 0.")

## K-nearest neighbours (KNN)

In [ ]:
print("\n1(d) K-nearest neighbours")
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.25, random_state=42, stratify=iris.target
)
k_values = [1, 3, 5, 7, 9]
knn_scores = []
for k in k_values:
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k, metric="minkowski", p=2))
    model.fit(X_train, y_train)
    score = accuracy_score(y_test, model.predict(X_test))
    knn_scores.append(score)
    print(f"k={k}: accuracy={score:.3f}")
best_k = k_values[int(np.argmax(knn_scores))]
print(f"Best k among tested values: {best_k}")
plt.figure(figsize=(5.5, 3.8))
plt.plot(k_values, knn_scores, marker="o")
plt.xticks(k_values)
plt.ylim(0, 1.05)
plt.xlabel("Number of neighbours (k)")
plt.ylabel("Test accuracy")
plt.title("Choosing k for KNN")
plt.tight_layout()
plt.show()

## Naive Bayes classification

In [ ]:
print("\n1(e) Gaussian Naive Bayes")
cancer = load_breast_cancer()
X = cancer.data
y = (cancer.target == 0).astype(int)  # 1 = malignant
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
nb_predictions = nb_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, nb_predictions):.3f}")
print(f"Malignant-class precision: {__import__('sklearn.metrics').metrics.precision_score(y_test, nb_predictions):.3f}")
print(f"Malignant-class recall: {__import__('sklearn.metrics').metrics.recall_score(y_test, nb_predictions):.3f}")
print(classification_report(y_test, nb_predictions, target_names=["benign", "malignant"], digits=3))
cm = confusion_matrix(y_test, nb_predictions)
plt.figure(figsize=(4.5, 3.8))
plt.imshow(cm, cmap="Blues")
plt.title("Gaussian Naive Bayes Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.xticks([0, 1], ["Benign", "Malignant"])
plt.yticks([0, 1], ["Benign", "Malignant"])
for (i, j), value in np.ndenumerate(cm):
    plt.text(j, i, str(value), ha="center", va="center")
plt.colorbar()
plt.tight_layout()
plt.show()